# WHO MONICA: Data Preparation

The objectives of the MONICA Newcastle Study were to measure the trends in coronary heart disease (CHD) morbidity and mortality and to assess the extent to which these trends are related to changes in known risk factors, daily living habits, health care, and major socioeconomic features measured at the same time in defined communities in different countries. [Source](https://www.maelstrom-research.org/study/monica-newcastle/)

The task is to predict the mortality outcome (`outcome`: dead/live).

**Version of record:** dataset `monica` from the R package **DAAG** (Maindonald & Braun, *Data Analysis and Graphics Using R*), distributed through [Rdatasets](https://vincentarelbundock.github.io/Rdatasets/). 6,367 rows, 11 features.

https://vincentarelbundock.github.io/Rdatasets/doc/DAAG/monica.html

Steps:
1. Load the raw data and encode the target (dead -> 1, live -> 0)
2. One-hot encode the categorical features 
3. Create a single, stratified hold-out test set
4. Generate imbalanced datasets (multiple IRs and repetitions) and size-matched control datasets
5. Scale and save all datasets

# 1. Dataset Configuration

In [1]:
DATASET_NAME = "monica"
RDATASETS_URL = "https://vincentarelbundock.github.io/Rdatasets/csv/DAAG/monica.csv"

In [2]:
import json
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample

notebook_dir = Path().resolve()
project_root = notebook_dir.parents[1]
sys.path.insert(0, str(project_root))

from config.config import get_config

config = get_config()

RAW_PATH = project_root / "data" / "raw" / f"{DATASET_NAME}.csv"
PROCESSED_PATH = project_root / "data" / "processed" / DATASET_NAME

ORIGINAL_TARGET = "outcome"
TARGET_FEATURE = "target"
NUMERICAL_FEATURES = ["age", "yronset"]
CATEGORICAL_FEATURES = ["sex", "hosp", "premi", "smstat", "diabetes", "highbp", "hichol", "angina", "stroke"]

RANDOM_STATE = config.experiment.random_state
IMBALANCE_RATIOS = config.experiment.imbalance_ratios
N_REPETITIONS = config.experiment.n_repetitions

RAW_PATH.parent.mkdir(parents=True, exist_ok=True)
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

print("Configuration:")
print(f"  - Dataset: {DATASET_NAME}")
print(f"  - Raw data path: {RAW_PATH}")
print(f"  - Processed data path: {PROCESSED_PATH}")
print(f"  - Target feature: {TARGET_FEATURE}")
print(f"  - Imbalance ratios: {IMBALANCE_RATIOS}")
print(f"  - Repetitions per IR: {N_REPETITIONS}")
print(f"  - Random state: {RANDOM_STATE}")

Configuration:
  - Dataset: monica
  - Raw data path: /home/adriano/Desktop/imbalance-effects-on-synthetic-data/data/raw/monica.csv
  - Processed data path: /home/adriano/Desktop/imbalance-effects-on-synthetic-data/data/processed/monica
  - Target feature: target
  - Imbalance ratios: [1, 3, 5, 7, 10, 20, 50, 100]
  - Repetitions per IR: 10
  - Random state: 42


In [3]:
df_raw = pd.read_csv(RDATASETS_URL).drop(columns=["rownames"])
df_raw.to_csv(RAW_PATH, index=False)

N_ORIGINAL_FEATURES = df_raw.shape[1] - 1
print(f"Raw dataset loaded and saved. Shape: {df_raw.shape}")
df_raw.head()

Raw dataset loaded and saved. Shape: (6367, 12)


,outcome,sex,age,yronset,premi,smstat,diabetes,highbp,hichol,angina,stroke,hosp
0,live,f,63,85,n,x,n,y,y,n,n,y
1,live,m,59,85,y,x,n,y,n,n,n,y
2,live,m,68,85,n,n,n,y,n,n,n,y
3,live,m,46,85,n,c,n,n,n,n,n,y
4,dead,m,48,85,n,n,y,n,n,y,n,y


# 2. Data Overview

Feature descriptions (DAAG documentation):
- `age`: age at onset; `yronset`: year of onset (85-93)
- `sex`: m/f; `hosp`: hospitalised y/n
- `premi`: previous myocardial infarction; `smstat`: smoking status (c current, x ex-smoker, n non-smoker)
- `diabetes`, `highbp`, `hichol`, `angina`, `stroke`: y/n

`nk` ("not known") marks unrecorded history. We kept it as its own category instead of treating it as missing.

In [4]:
print("Data types:")
print(df_raw.dtypes)
print(f"\nMissing values: {df_raw.isna().sum().sum()}")
print(f"Duplicate rows: {df_raw.duplicated().sum()}")

for col in [ORIGINAL_TARGET] + CATEGORICAL_FEATURES:
    print(f"  {col}: {df_raw[col].value_counts().to_dict()}")

df_raw[NUMERICAL_FEATURES].describe()

Data types:
outcome     object
sex         object
age          int64
yronset      int64
premi       object
smstat      object
diabetes    object
highbp      object
hichol      object
angina      object
stroke      object
hosp        object
dtype: object

Missing values: 0
Duplicate rows: 754
  outcome: {'live': 3525, 'dead': 2842}
  sex: {'m': 4605, 'f': 1762}
  hosp: {'y': 4442, 'n': 1925}
  premi: {'n': 4122, 'y': 1511, 'nk': 734}
  smstat: {'c': 2051, 'x': 1938, 'n': 1460, 'nk': 918}
  diabetes: {'n': 4664, 'nk': 885, 'y': 818}
  highbp: {'y': 2877, 'n': 2542, 'nk': 948}
  hichol: {'n': 3294, 'y': 1840, 'nk': 1233}
  angina: {'n': 3473, 'y': 1919, 'nk': 975}
  stroke: {'n': 4881, 'nk': 926, 'y': 560}


,age,yronset
count,6367.000000,6367.000000
mean,59.419978,88.749018
std,7.853923,2.558180
min,35.000000,85.000000
25%,55.000000,87.000000
50%,61.000000,89.000000
75%,66.000000,91.000000
max,69.000000,93.000000


# 3. Target and Categorical Encoding

- Target: **dead -> 1**, **live -> 0**
- Categorical features are one-hot encoded (`drop_first=False`, as in the other datasets)

In [5]:
df_cleaned = df_raw.copy()
df_cleaned[TARGET_FEATURE] = df_cleaned[ORIGINAL_TARGET].map({'dead': 1, 'live': 0}).astype(int)
df_cleaned = df_cleaned.drop(columns=[ORIGINAL_TARGET])
df_cleaned[NUMERICAL_FEATURES] = df_cleaned[NUMERICAL_FEATURES].astype(float)

df_encoded = pd.get_dummies(df_cleaned, columns=CATEGORICAL_FEATURES, drop_first=False, dtype=float)

ALL_FEATURES = [c for c in df_encoded.columns if c != TARGET_FEATURE]
print(f"Shape after encoding: {df_encoded.shape}")
print(f"Total features: {len(ALL_FEATURES)}")
print(f"  - Numerical: {len(NUMERICAL_FEATURES)}")
print(f"  - One-hot categorical: {len(ALL_FEATURES) - len(NUMERICAL_FEATURES)}")
df_encoded.head()

Shape after encoding: (6367, 29)
Total features: 28
  - Numerical: 2
  - One-hot categorical: 26


,age,yronset,target,sex_f,sex_m,hosp_n,hosp_y,premi_n,premi_nk,premi_y,...,highbp_y,hichol_n,hichol_nk,hichol_y,angina_n,angina_nk,angina_y,stroke_n,stroke_nk,stroke_y
0,63.0,85.0,0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,...,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
1,59.0,85.0,0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,...,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
2,68.0,85.0,0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,...,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
3,46.0,85.0,0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
4,48.0,85.0,1,0.0,1.0,0.0,1.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0


# Confirming Majority and Minority Classes

The majority and minority classes are derived from the class counts rather than hard-coded.
- **Class 1:** dead
- **Class 0:** live

In [6]:
class_counts = df_encoded[TARGET_FEATURE].value_counts().sort_index()
print("Target variable distribution:")
print(class_counts)

CLASS_LABELS = {1: "dead", 0: "live"}
CLASS_MAJORITY = class_counts.idxmax()
CLASS_MINORITY = class_counts.idxmin()
assert CLASS_MAJORITY != CLASS_MINORITY, "Classes are perfectly balanced; choose the minority explicitly."

for cls, n in class_counts.items():
    print(f"  - {CLASS_LABELS[cls]} ({cls}): {n:,} ({n / len(df_encoded) * 100:.2f}%)")

print(f"\nMajority class: {CLASS_LABELS[CLASS_MAJORITY]} ({CLASS_MAJORITY})")
print(f"Minority class: {CLASS_LABELS[CLASS_MINORITY]} ({CLASS_MINORITY})")

natural_ir = class_counts.max() / class_counts.min()
print(f"\nNatural imbalance ratio: {natural_ir:.2f}:1")

Target variable distribution:
target
0    3525
1    2842
Name: count, dtype: int64
  - live (0): 3,525 (55.36%)
  - dead (1): 2,842 (44.64%)

Majority class: live (0)
Minority class: dead (1)

Natural imbalance ratio: 1.24:1


# Create a Hold-Out Test Set

We perform a one-time stratified split to create a final test set. All experimental datasets will be generated from the `train_full_df`.

In [7]:
X = df_encoded.drop(TARGET_FEATURE, axis=1)
y = df_encoded[[TARGET_FEATURE]]

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y,
    test_size=config.experiment.test_size,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"Full training set shape: {X_train_full.shape}")
print(f"Hold-out test set shape: {X_test.shape}")

Full training set shape: (5093, 28)
Hold-out test set shape: (1274, 28)


In [8]:
train_full_df = pd.concat([X_train_full, y_train_full], axis=1)
print("Training set class distribution:")
print(train_full_df[TARGET_FEATURE].value_counts().sort_index())

Training set class distribution:
target
0    2820
1    2273
Name: count, dtype: int64


# Generate Imbalanced and Control Datasets with Multiple Repetitions

For each IR, we create multiple repetitions by sampling different subsets of the minority class.

**Methodology:**
1. Start with the **full majority class**.
2. **Undersample the minority class** to achieve the desired Imbalance Ratio (IR). For IR=1, the majority is undersampled to the minority size instead.
3. **Repeat this sampling N_REPETITIONS times** with different random seeds.
4. Create a size-matched **control dataset** (original class ratio) for each IR and repetition.

In [9]:
minority_df = train_full_df[train_full_df[TARGET_FEATURE] == CLASS_MINORITY]
majority_df = train_full_df[train_full_df[TARGET_FEATURE] == CLASS_MAJORITY]
n_minority_available = len(minority_df)
n_majority_available = len(majority_df)

print("Full training set composition:")
print(f"  - Majority ({CLASS_LABELS[CLASS_MAJORITY]}): {n_majority_available:,}")
print(f"  - Minority ({CLASS_LABELS[CLASS_MINORITY]}): {n_minority_available:,}")
print(f"\nGenerating datasets with {N_REPETITIONS} repetitions per imbalance ratio...")

generated_datasets = {}
skipped_ratios = []

for ir in IMBALANCE_RATIOS:
    print(f"\nProcessing Imbalance Ratio (IR) = {ir}:1")

    for rep_id in range(1, N_REPETITIONS + 1):
        rep_seed = RANDOM_STATE + (ir * 1000) + rep_id

        if ir == 1:
            # For IR=1:1, undersample majority to match minority
            majority_undersampled = resample(
                majority_df,
                replace=False,
                n_samples=n_minority_available,
                random_state=rep_seed
            )
            imbalanced_df = pd.concat([majority_undersampled, minority_df])

        else:
            # For IR > 1, keep full majority and undersample minority
            n_minority_imbalanced = int(n_majority_available / ir)

            if n_minority_imbalanced > n_minority_available:
                if rep_id == 1:
                    print(f"  SKIPPING IR={ir}: Requires {n_minority_imbalanced} minority samples but only {n_minority_available} available.")
                    skipped_ratios.append(ir)
                continue
            if n_minority_imbalanced < 1:
                if rep_id == 1:
                    print(f"  SKIPPING IR={ir}: Would result in zero minority samples.")
                    skipped_ratios.append(ir)
                continue

            minority_undersampled = resample(
                minority_df,
                replace=False,
                n_samples=n_minority_imbalanced,
                random_state=rep_seed
            )
            imbalanced_df = pd.concat([majority_df, minority_undersampled])

        total_size = len(imbalanced_df)
        generated_datasets[f'imbalanced_ir_{ir}_rep{rep_id}'] = imbalanced_df

        if rep_id == 1:
            n_maj = (imbalanced_df[TARGET_FEATURE] == CLASS_MAJORITY).sum()
            n_min = (imbalanced_df[TARGET_FEATURE] == CLASS_MINORITY).sum()
            print(f"  Rep {rep_id}: {total_size:,} samples ({n_maj:,} majority, {n_min:,} minority)")

        # Create control dataset with original class ratio
        if total_size >= len(train_full_df):
            control_df = train_full_df.copy()
        else:
            control_df, _ = train_test_split(
                train_full_df,
                train_size=total_size,
                random_state=rep_seed,
                stratify=train_full_df[TARGET_FEATURE]
            )
        generated_datasets[f'control_ir_{ir}_rep{rep_id}'] = control_df

print(f"\nTotal datasets created: {len(generated_datasets)}")
print(f"  - Imbalanced: {len([k for k in generated_datasets if 'imbalanced' in k])}")
print(f"  - Control: {len([k for k in generated_datasets if 'control' in k])}")

if skipped_ratios:
    print(f"\nSkipped imbalance ratios (insufficient minority samples): {sorted(set(skipped_ratios))}")

Full training set composition:
  - Majority (live): 2,820
  - Minority (dead): 2,273

Generating datasets with 10 repetitions per imbalance ratio...

Processing Imbalance Ratio (IR) = 1:1
  Rep 1: 4,546 samples (2,273 majority, 2,273 minority)



Processing Imbalance Ratio (IR) = 3:1
  Rep 1: 3,760 samples (2,820 majority, 940 minority)



Processing Imbalance Ratio (IR) = 5:1
  Rep 1: 3,384 samples (2,820 majority, 564 minority)



Processing Imbalance Ratio (IR) = 7:1
  Rep 1: 3,222 samples (2,820 majority, 402 minority)



Processing Imbalance Ratio (IR) = 10:1
  Rep 1: 3,102 samples (2,820 majority, 282 minority)



Processing Imbalance Ratio (IR) = 20:1
  Rep 1: 2,961 samples (2,820 majority, 141 minority)



Processing Imbalance Ratio (IR) = 50:1
  Rep 1: 2,876 samples (2,820 majority, 56 minority)

Processing Imbalance Ratio (IR) = 100:1
  Rep 1: 2,848 samples (2,820 majority, 28 minority)



Total datasets created: 160
  - Imbalanced: 80
  - Control: 80


In [10]:
summary_data = []
for name in sorted(generated_datasets.keys()):
    df_temp = generated_datasets[name]
    n_maj = (df_temp[TARGET_FEATURE] == CLASS_MAJORITY).sum()
    n_min = (df_temp[TARGET_FEATURE] == CLASS_MINORITY).sum()
    actual_ir = n_maj / n_min if n_min > 0 else float('inf')
    summary_data.append({
        'Dataset': name,
        'Total': len(df_temp),
        'Majority': n_maj,
        'Minority': n_min,
        'Actual IR': f"{actual_ir:.2f}:1"
    })

summary_df = pd.DataFrame(summary_data)
display(summary_df[summary_df['Dataset'].str.endswith('_rep1')])

,Dataset,Total,Majority,Minority,Actual IR
0,control_ir_100_rep1,2848,1577,1271,1.24:1
10,control_ir_10_rep1,3102,1718,1384,1.24:1
20,control_ir_1_rep1,4546,2517,2029,1.24:1
30,control_ir_20_rep1,2961,1640,1321,1.24:1
40,control_ir_3_rep1,3760,2082,1678,1.24:1
50,control_ir_50_rep1,2876,1592,1284,1.24:1
60,control_ir_5_rep1,3384,1874,1510,1.24:1
70,control_ir_7_rep1,3222,1784,1438,1.24:1
80,imbalanced_ir_100_rep1,2848,2820,28,100.71:1
90,imbalanced_ir_10_rep1,3102,2820,282,10.00:1


# Preprocessing and Saving All Datasets

We apply StandardScaler to all features. The scaler is fitted **once** on the full training data, and the same scaler transforms every generated training set and the hold-out test set.

In [11]:
FEATURES_TO_SCALE = ALL_FEATURES

scaler = StandardScaler()
scaler.fit(train_full_df[FEATURES_TO_SCALE])

print(f"Scaler fitted on full training data ({len(FEATURES_TO_SCALE)} features).")
for feat in NUMERICAL_FEATURES:
    idx = FEATURES_TO_SCALE.index(feat)
    print(f"  {feat}: mean={scaler.mean_[idx]:.4f}, std={scaler.scale_[idx]:.4f}")

Scaler fitted on full training data (28 features).
  age: mean=59.4481, std=7.8311
  yronset: mean=88.7251, std=2.5515


In [12]:
def scale_and_attach_target(df):
    X_scaled = pd.DataFrame(scaler.transform(df[FEATURES_TO_SCALE]), columns=FEATURES_TO_SCALE)
    X_scaled[TARGET_FEATURE] = df[TARGET_FEATURE].to_numpy()
    return X_scaled


print("Scaling and saving datasets...\n")
for saved_count, (name, df) in enumerate(generated_datasets.items(), start=1):
    scale_and_attach_target(df).to_csv(PROCESSED_PATH / f"train_{name}.csv", index=False)
    if saved_count % 20 == 0:
        print(f"  Saved {saved_count} files...")

test_full_df = pd.concat([X_test, y_test], axis=1)
test_df = scale_and_attach_target(test_full_df)
test_df.to_csv(PROCESSED_PATH / "test.csv", index=False)

print(f"\nSaved all {len(generated_datasets)} training files.")
print(f"Saved test set: test.csv ({len(test_df):,} samples)")
print(test_df[TARGET_FEATURE].value_counts().sort_index())

Scaling and saving datasets...



  Saved 20 files...


  Saved 40 files...


  Saved 60 files...


  Saved 80 files...


  Saved 100 files...


  Saved 120 files...


  Saved 140 files...


  Saved 160 files...

Saved all 160 training files.
Saved test set: test.csv (1,274 samples)
target
0    705
1    569
Name: count, dtype: int64


# Save Metadata

In [13]:
NOTES = (
    "WHO MONICA (Newcastle centre) myocardial infarction outcomes from R package DAAG via Rdatasets. "
    "'nk' (not known) kept as its own category; all categorical features one-hot encoded; duplicates kept."
)

In [14]:
created_irs = sorted({int(k.split('_')[2]) for k in generated_datasets if 'imbalanced' in k})

metadata = {
    "dataset_name": DATASET_NAME,
    "source": "R package DAAG (dataset 'monica'), WHO MONICA project, Newcastle centre; via Rdatasets",
    "source_url": RDATASETS_URL,
    "original_target_encoding": {"dead": "1", "live": "0"},
    "missing_value_handling": {"strategy": "none needed; 'nk' kept as a category"},
    "target_feature": TARGET_FEATURE,
    "processing_timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "n_train_files": len(generated_datasets),
    "imbalance_ratios_requested": IMBALANCE_RATIOS,
    "imbalance_ratios_created": created_irs,
    "n_repetitions": N_REPETITIONS,
    "random_state": RANDOM_STATE,
    "test_size": len(test_df),
    "train_full_size": len(train_full_df),
    "n_original_features": N_ORIGINAL_FEATURES,
    "n_features_after_encoding": len(FEATURES_TO_SCALE),
    "numerical_features": NUMERICAL_FEATURES,
    "categorical_features_original": CATEGORICAL_FEATURES,
    "all_features": FEATURES_TO_SCALE,
    "class_mapping": {
        str(CLASS_MAJORITY): f"{CLASS_LABELS[CLASS_MAJORITY]} (majority)",
        str(CLASS_MINORITY): f"{CLASS_LABELS[CLASS_MINORITY]} (minority)",
    },
    "natural_imbalance_ratio": round(float(natural_ir), 2),
    "notes": NOTES,
}

metadata_path = PROCESSED_PATH / "metadata.json"
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Metadata saved to: {metadata_path}")
for key, value in metadata.items():
    if key not in ['all_features', 'numerical_features', 'categorical_features_original']:
        print(f"  {key}: {value}")

Metadata saved to: /home/adriano/Desktop/imbalance-effects-on-synthetic-data/data/processed/monica/metadata.json
  dataset_name: monica
  source: R package DAAG (dataset 'monica'), WHO MONICA project, Newcastle centre; via Rdatasets
  source_url: https://vincentarelbundock.github.io/Rdatasets/csv/DAAG/monica.csv
  original_target_encoding: {'dead': '1', 'live': '0'}
  missing_value_handling: {'strategy': "none needed; 'nk' kept as a category"}
  target_feature: target
  processing_timestamp: 2026-11-18 21:31:39
  n_train_files: 160
  imbalance_ratios_requested: [1, 3, 5, 7, 10, 20, 50, 100]
  imbalance_ratios_created: [1, 3, 5, 7, 10, 20, 50, 100]
  n_repetitions: 10
  random_state: 42
  test_size: 1274
  train_full_size: 5093
  n_original_features: 11
  n_features_after_encoding: 28
  class_mapping: {'0': 'live (majority)', '1': 'dead (minority)'}
  natural_imbalance_ratio: 1.24
  notes: WHO MONICA (Newcastle centre) myocardial infarction outcomes from R package DAAG via Rdatasets. 'n

In [15]:
saved_files = list(PROCESSED_PATH.glob("*.csv"))
print(f"  - Total CSV files saved: {len(saved_files)}")
print(f"  - Expected: {len(generated_datasets) + 1} (training + test)")
total_size_mb = sum(f.stat().st_size for f in saved_files) / (1024 * 1024)
print(f"  - Total size: {total_size_mb:.2f} MB")

sample_file = PROCESSED_PATH / "train_imbalanced_ir_1_rep1.csv"
sample_df = pd.read_csv(sample_file)
print(f"\nSample verification ({sample_file.name}):")
print(f"  Shape: {sample_df.shape}")
print(f"  Missing values: {sample_df.isna().sum().sum()}")
print(f"  Columns match test set: {list(sample_df.columns) == list(test_df.columns)}")
print(sample_df[TARGET_FEATURE].value_counts().sort_index())
sample_df[NUMERICAL_FEATURES[:3]].describe().round(3)

  - Total CSV files saved: 161
  - Expected: 161 (training + test)
  - Total size: 281.55 MB

Sample verification (train_imbalanced_ir_1_rep1.csv):
  Shape: (4546, 29)
  Missing values: 0
  Columns match test set: True
target
0    2273
1    2273
Name: count, dtype: int64


,age,yronset
count,4546.000,4546.000
mean,0.023,-0.006
std,0.989,0.999
min,-3.122,-1.460
25%,-0.568,-0.676
50%,0.326,0.108
75%,0.837,0.892
max,1.220,1.675
